# 02: Preprocess APTOS 2019

Crops the 3,662 labelled APTOS 2019 images (Aravind Eye Hospital, India) to the fundus, resizes
them to 512 px and writes them as JPEG into zip shards, plus a validated manifest. APTOS is used
only as an external test set. Then looks for near-duplicates within APTOS and between APTOS and
EyePACS: a phash-close pair counts as a duplicate when its structure similarity is at least the
threshold calibrated on EyePACS.

**Settings** (right-hand panel): Accelerator **None**, Internet **On**.
**Inputs** (*Add Input*):
- *Competitions* → **APTOS 2019 Blindness Detection**
- *Notebooks* → *Your Work* → **01: Preprocess EyePACS** (its saved output)

**Run:** *Save Version* → *Save & Run All (Commit)*. Takes about 30–45 min.

**Output** (`/kaggle/working/aptos512/`):
- `aptos_000.zip`, `aptos_001.zip`: processed images (`{id_code}.jpg`), with `.records.parquet`
- `manifest_aptos.parquet`: labels + image columns, schema-validated
- `aptos_duplicate_candidates.parquet`: phash-close pairs involving APTOS, with similarity
- `failures.csv`, `summary.json`, `spot_check.png`, `aptos_top_pairs.png`

> RetinaGuard is a research prototype, **not a medical device**.

In [ ]:
REPO_URL = "https://github.com/FidaMarzouk/retinaguard.git"
GIT_REF = "main"
OUT_DIR = "/kaggle/working/aptos512"
SEED = 0

In [ ]:
%pip install -q "retinaguard @ git+{REPO_URL}@{GIT_REF}" "pandera>=0.20"

In [ ]:
# --- 1. Locate APTOS, its labels and the EyePACS output ----------------------
import subprocess
import time
from pathlib import Path

import pandas as pd

from retinaguard import __version__
from retinaguard.data.sources import aptos

INPUT_ROOT = Path("/kaggle/input")
labels_found = [p for p in INPUT_ROOT.glob("**/train.csv") if (p.parent / "train_images").is_dir()]
assert labels_found, "train.csv + train_images/ not found: attach the APTOS 2019 competition"
labels_path = labels_found[0]
eyepacs_found = sorted(INPUT_ROOT.glob("**/manifest_eyepacs.parquet"))
assert eyepacs_found, "manifest_eyepacs.parquet not found: attach the output of notebook 01"
eyepacs_dir = eyepacs_found[0].parent

git_sha = subprocess.run(
    ["git", "ls-remote", REPO_URL, GIT_REF], capture_output=True, text=True
).stdout.split("\t")[0]
print(f"retinaguard {__version__} @ {git_sha[:10]}")

images = aptos.KaggleImages(labels_path.parent)
labels = aptos.read_labels(labels_path)
missing = set(labels["image_id"].str.removeprefix("aptos:")) - images.names()
print(f"{len(images.names()):,} images, {len(labels):,} labels")
print("grades:", labels["dr_grade"].value_counts().sort_index().to_dict())
assert not missing, f"{len(missing)} labelled images missing, e.g. {sorted(missing)[:5]}"

In [ ]:
# --- 2. Process all images, shard by shard (resumable) -------------------------
from retinaguard.data.processing import build_shard, shard_ids

out_dir = Path(OUT_DIR)
shards = shard_ids(labels["image_id"])
all_records = []
start = time.perf_counter()
for i, ids in enumerate(shards):
    t = time.perf_counter()
    records = build_shard(out_dir, f"aptos_{i:03d}.zip", ids, images.read)
    all_records.append(records)
    print(
        f"shard {i + 1}/{len(shards)}: {len(ids)} images, "
        f"{records['error'].notna().sum()} failed, {time.perf_counter() - t:5.0f}s",
        flush=True,
    )
records = pd.concat(all_records, ignore_index=True)

In [ ]:
# --- 3. Manifest, failures and summary ---------------------------------------
import json

from retinaguard.data.invariants import duplicate_image_ids
from retinaguard.data.processing import attach_records

manifest = attach_records(labels, records)
assert not duplicate_image_ids(manifest)
manifest.to_parquet(out_dir / "manifest_aptos.parquet", index=False)

failures = records[records["error"].notna()]
failures.to_csv(out_dir / "failures.csv", index=False)

ok = records[records["error"].isna()]
radius_ratio = 2 * ok["fundus_radius"] / ok["width"]
sizes = (ok["width"].astype(str) + "x" + ok["height"].astype(str)).value_counts()
summary = {
    "retinaguard_version": __version__,
    "git_sha": git_sha,
    "preprocess_version": str(records["preprocess_version"].iloc[0]),
    "labelled_images": len(labels),
    "processed": len(ok),
    "failed": len(failures),
    "failure_reasons": failures["error"].str.split(":").str[0].value_counts().to_dict(),
    "exact_duplicate_raw_files": int(records["sha256"].duplicated().sum()),
    "raw_sizes_top10": sizes.head(10).to_dict(),
    "radius_ratio_p01_p50_p99": [
        round(float(q), 3) for q in radius_ratio.quantile([0.01, 0.5, 0.99])
    ],
    "minutes": round((time.perf_counter() - start) / 60, 1),
}
(out_dir / "summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print(failures[["image_id", "error"]].head(20).to_string())

In [ ]:
# --- 4. Spot check: a few processed images straight from the shards ------------
import matplotlib.pyplot as plt

from retinaguard.data.preprocess import local_mean_normalise
from retinaguard.data.processing import ProcessedImages, shard_roots

done = manifest.dropna(subset=["processed_path"])
paths = dict(zip(done["image_id"], done["processed_path"], strict=True))
eyepacs = pd.read_parquet(eyepacs_found[0]).dropna(subset=["processed_path"])
paths.update(zip(eyepacs["image_id"], eyepacs["processed_path"], strict=True))
reader = ProcessedImages(paths, shard_roots(out_dir, eyepacs_dir))

sample = done.sample(4, random_state=SEED)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for col, (_, row) in enumerate(sample.iterrows()):
    img = reader.read(row["image_id"])
    for ax, shown in zip(axes[:, col], (img, local_mean_normalise(img)), strict=True):
        ax.imshow(shown)
        ax.set_title(f"{row['image_id']} grade {row['dr_grade']}", fontsize=8)
        ax.axis("off")
plt.tight_layout()
fig.savefig(out_dir.parent / "spot_check.png", dpi=80)
plt.show()

In [ ]:
# --- 5. Near-duplicates within APTOS and between APTOS and EyePACS -------------
from functools import cache

import numpy as np

from retinaguard.data.duplicates import (
    CANDIDATE_MAX_DISTANCE,
    DUPLICATE_MIN_SIMILARITY,
    candidate_frame,
    score_candidates,
    structure_signature,
    structure_similarity,
)


@cache
def signature(image_id):
    return structure_signature(reader.read(image_id))


both = pd.concat([done[["image_id", "phash"]], eyepacs[["image_id", "phash"]]])
t0 = time.perf_counter()
cand = candidate_frame(both["image_id"].tolist(), both["phash"].tolist(), involving="aptos")
cand["kind"] = np.where(
    cand["image_id_a"].str.startswith("aptos:") & cand["image_id_b"].str.startswith("aptos:"),
    "aptos-aptos",
    "aptos-eyepacs",
)
cand = score_candidates(cand, signature)
print(
    f"{len(cand):,} candidates (phash <= {CANDIDATE_MAX_DISTANCE}) scored in "
    f"{time.perf_counter() - t0:.0f}s"
)

# Sanity check of the threshold on this site: random APTOS-EyePACS pairs are different eyes.
rng = np.random.default_rng(SEED)
a = rng.choice(done["image_id"].to_numpy(), 2000)
b = rng.choice(eyepacs["image_id"].to_numpy(), 2000)
random_pairs = np.array(
    [structure_similarity(signature(x), signature(y)) for x, y in zip(a, b, strict=True)]
)
print(
    "random APTOS-EyePACS similarity min/p50/p99/max:",
    [round(float(v), 3) for v in np.quantile(random_pairs, [0, 0.5, 0.99, 1])],
)

confirmed = cand[cand["similarity"] >= DUPLICATE_MIN_SIMILARITY]
print(
    f"confirmed duplicates (similarity >= {DUPLICATE_MIN_SIMILARITY}):",
    confirmed["kind"].value_counts().to_dict(),
)

grade = dict(zip(both["image_id"], pd.concat([done["dr_grade"], eyepacs["dr_grade"]]), strict=True))
confirmed = confirmed.assign(
    grade_a=confirmed["image_id_a"].map(grade), grade_b=confirmed["image_id_b"].map(grade)
)
for kind, rows in confirmed.groupby("kind"):
    differ = rows["grade_a"] != rows["grade_b"]
    print(f"{kind}: {len(rows)} pairs, {int(differ.sum())} with different grades")
cand.to_parquet(out_dir / "aptos_duplicate_candidates.parquet", index=False)
summary["duplicates"] = {
    "candidates": cand["kind"].value_counts().to_dict(),
    "confirmed": confirmed["kind"].value_counts().to_dict(),
    "random_aptos_eyepacs_max": round(float(random_pairs.max()), 3),
}
_ = (out_dir / "summary.json").write_text(json.dumps(summary, indent=2))

In [ ]:
# --- 6. Look at the most similar pairs of each kind ----------------------------
top = cand.sort_values("similarity", ascending=False).groupby("kind").head(4)
if len(top):
    fig, axes = plt.subplots(len(top), 2, figsize=(6, 3 * len(top)), squeeze=False)
    for row, (_, pair) in zip(axes, top.iterrows(), strict=True):
        for ax, image_id in zip(row, (pair["image_id_a"], pair["image_id_b"]), strict=True):
            ax.imshow(reader.read(image_id))
            ax.set_title(f"{image_id}  grade {grade[image_id]}", fontsize=8)
            ax.axis("off")
        row[0].text(
            0,
            -0.12,
            f"{pair['kind']}: similarity {pair['similarity']:.3f}, "
            f"phash distance {pair['phash_distance']}",
            transform=row[0].transAxes,
            fontsize=8,
        )
    plt.tight_layout()
    fig.savefig(out_dir.parent / "aptos_top_pairs.png", dpi=80)
    plt.show()

**After the version finishes:** download `aptos512/manifest_aptos.parquet` and
`aptos512/aptos_duplicate_candidates.parquet` from the **Output** tab into
`data/interim/aptos/`, and send back the printed summary, the duplicate counts and
`aptos_top_pairs.png`.